In [2]:
import requests
import pandas as pd

# Your Census API key
api_key = "861cf720589cf04f001ee676f65ede640f5bb7b1"

# Attempting the 2024 dataset
url = "https://api.census.gov/data/2024/acs/acs5"
params = {
    "get": "NAME,B01001_001E,B19013_001E,B17001_001E,B17001_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B01002_001E,B03002_001E,B03002_003E,B03002_004E,B03002_005E,B03002_006E,B03002_007E,B03002_008E,B03002_009E,B03002_012E,B01001_002E,B01001_026E",
    "for": "zip code tabulation area:*"
}

if api_key:
    params["key"] = api_key

print("Fetching data from US Census Bureau...")
response = requests.get(url, params=params)

if response.status_code == 200:
    try:
        data = response.json()
        # Convert response to DataFrame (first row contains headers)
        df = pd.DataFrame(data[1:], columns=data[0])

        # The ZIP code column is typically named 'zip code tabulation area'
        zip_col = 'zip code tabulation area'

        if zip_col in df.columns:
            # Massachusetts zip codes generally start with 010 through 027
            df[zip_col] = df[zip_col].astype(str).str.zfill(5)

            # Filter for Massachusetts ZCTAs (ZIP codes starting with '01' or '02')
            ma_df = df[df[zip_col].str.startswith(('01', '02'))].copy()

            print(f"Successfully fetched {len(df)} total ZIP codes.")
            print(f"Filtered down to {len(ma_df)} Massachusetts ZIP codes.")

            # Display the first few rows of Massachusetts data
            display(ma_df.head())
        else:
            print(f"Error: Column '{zip_col}' not found in the response. Available columns:", df.columns)
            display(df.head())
    except ValueError as e:
        print("Failed to parse JSON response. The server returned raw text instead of JSON:")
        print(response.text)
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")
    print(response.text)

Fetching data from US Census Bureau...
Successfully fetched 33772 total ZIP codes.
Filtered down to 620 Massachusetts ZIP codes.


,NAME,B01001_001E,B19013_001E,B17001_001E,B17001_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,...,B03002_004E,B03002_005E,B03002_006E,B03002_007E,B03002_008E,B03002_009E,B03002_012E,B01001_002E,B01001_026E,zip code tabulation area
132,ZCTA5 01001,16234,75342,15762,984,12443,2667,1313,205,264,...,678,0,498,0,50,400,1073,8180,8054,01001
133,ZCTA5 01002,26370,70818,23164,5240,13824,3215,4400,416,2392,...,1178,5,2597,0,31,1679,2692,13133,13237,01002
134,ZCTA5 01003,13510,-666666666,90,19,159,7,45,0,23,...,751,0,2870,0,0,877,849,6178,7332,01003
135,ZCTA5 01005,5204,111148,5097,446,3928,719,456,44,103,...,100,0,0,0,0,69,299,2544,2660,01005
136,ZCTA5 01007,15419,108125,15376,1309,10927,2444,1981,333,411,...,314,37,331,0,196,508,1109,7282,8137,01007


In [3]:
import numpy as np

# Create a working copy of our Massachusetts DataFrame
cleaned_ma_df = ma_df.copy()

# Identify columns to convert to numeric (all except NAME and the ZIP code column)
exclude_cols = ['NAME', 'zip code tabulation area']
numeric_cols = [col for col in cleaned_ma_df.columns if col not in exclude_cols]

for col in numeric_cols:
    # Convert column to numeric, forcing errors to NaN if there are corrupted strings
    cleaned_ma_df[col] = pd.to_numeric(cleaned_ma_df[col], errors='coerce')

    # Replace Census negative sentinel values (like -666666666) with actual NaNs
    # We target any extremely large negative numbers used by the Census Bureau
    cleaned_ma_df.loc[cleaned_ma_df[col] <= -1000000, col] = np.nan

# Summary of missing values (NaNs) in each column after cleaning
missing_summary = cleaned_ma_df[numeric_cols].isna().sum().to_frame(name='NaN_Count')
missing_summary['NaN_Percentage'] = (missing_summary['NaN_Count'] / len(cleaned_ma_df)) * 100

print("Summary of missing values (NaNs) after cleaning:")
display(missing_summary)

print("\nPreview of the cleaned Massachusetts dataset:")
display(cleaned_ma_df.head())

Summary of missing values (NaNs) after cleaning:


,NaN_Count,NaN_Percentage
B01001_001E,0,0.000000
B19013_001E,40,6.451613
B17001_001E,0,0.000000
B17001_002E,0,0.000000
B15003_001E,0,0.000000
B15003_022E,0,0.000000
B15003_023E,0,0.000000
B15003_024E,0,0.000000
B15003_025E,0,0.000000
B01002_001E,11,1.774194



Preview of the cleaned Massachusetts dataset:


,NAME,B01001_001E,B19013_001E,B17001_001E,B17001_002E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,...,B03002_004E,B03002_005E,B03002_006E,B03002_007E,B03002_008E,B03002_009E,B03002_012E,B01001_002E,B01001_026E,zip code tabulation area
132,ZCTA5 01001,16234.0,75342.0,15762.0,984.0,12443.0,2667.0,1313.0,205.0,264.0,...,678.0,0.0,498.0,0.0,50.0,400.0,1073.0,8180.0,8054.0,01001
133,ZCTA5 01002,26370.0,70818.0,23164.0,5240.0,13824.0,3215.0,4400.0,416.0,2392.0,...,1178.0,5.0,2597.0,0.0,31.0,1679.0,2692.0,13133.0,13237.0,01002
134,ZCTA5 01003,13510.0,NaN,90.0,19.0,159.0,7.0,45.0,0.0,23.0,...,751.0,0.0,2870.0,0.0,0.0,877.0,849.0,6178.0,7332.0,01003
135,ZCTA5 01005,5204.0,111148.0,5097.0,446.0,3928.0,719.0,456.0,44.0,103.0,...,100.0,0.0,0.0,0.0,0.0,69.0,299.0,2544.0,2660.0,01005
136,ZCTA5 01007,15419.0,108125.0,15376.0,1309.0,10927.0,2444.0,1981.0,333.0,411.0,...,314.0,37.0,331.0,0.0,196.0,508.0,1109.0,7282.0,8137.0,01007


In [4]:
# Initialize a new DataFrame for clean analytics
final_df = pd.DataFrame()

# Keep the index, geographic name, and ZIP code
final_df['NAME'] = cleaned_ma_df['NAME']
final_df['zip_code'] = cleaned_ma_df['zip code tabulation area']

# 1. Total Population
final_df['total_population'] = cleaned_ma_df['B01001_001E']

# 2. Median Household Income
final_df['median_household_income'] = cleaned_ma_df['B19013_001E']

# 3. Poverty Rate Variable Calculation
# B17001_002E: Income below poverty level / B17001_001E: Poverty status universe (denominator)
final_df['poverty_rate'] = cleaned_ma_df['B17001_002E'] / cleaned_ma_df['B17001_001E']

# 4. Educational Attainment (Bachelor's or Higher)
# Sum of degrees (Bachelor + Master + Professional + Doctorate) / Total population 25+ (denominator)
degrees_sum = (
    cleaned_ma_df['B15003_022E'] +
    cleaned_ma_df['B15003_023E'] +
    cleaned_ma_df['B15003_024E'] +
    cleaned_ma_df['B15003_025E']
)
final_df['pct_bachelors_or_higher'] = degrees_sum / cleaned_ma_df['B15003_001E']

# 5. Median Age
final_df['median_age'] = cleaned_ma_df['B01002_001E']

# 6. Racial & Ethnic Composition (divided by B03002_001E total)
race_denominator = cleaned_ma_df['B03002_001E']
final_df['pct_white_non_hispanic'] = cleaned_ma_df['B03002_003E'] / race_denominator
final_df['pct_black_non_hispanic'] = cleaned_ma_df['B03002_004E'] / race_denominator
final_df['pct_native_american_non_hispanic'] = cleaned_ma_df['B03002_005E'] / race_denominator
final_df['pct_asian_non_hispanic'] = cleaned_ma_df['B03002_006E'] / race_denominator
final_df['pct_pacific_islander_non_hispanic'] = cleaned_ma_df['B03002_007E'] / race_denominator
final_df['pct_other_race_non_hispanic'] = cleaned_ma_df['B03002_008E'] / race_denominator
final_df['pct_two_or_more_races_non_hispanic'] = cleaned_ma_df['B03002_009E'] / race_denominator
final_df['pct_hispanic_or_latino'] = cleaned_ma_df['B03002_012E'] / race_denominator

# 7. Sex Demographics (Males per 100 Females)
# Sex Ratio: (Total Males / Total Females) * 100
final_df['sex_ratio'] = (cleaned_ma_df['B01001_002E'] / cleaned_ma_df['B01001_026E']) * 100

# Alternative Sex Percentages
final_df['pct_male'] = cleaned_ma_df['B01001_002E'] / cleaned_ma_df['B01001_001E']
final_df['pct_female'] = cleaned_ma_df['B01001_026E'] / cleaned_ma_df['B01001_001E']

print("Created cleaned analytics DataFrame 'final_df'!")
display(final_df.head())

Created cleaned analytics DataFrame 'final_df'!


,NAME,zip_code,total_population,median_household_income,poverty_rate,pct_bachelors_or_higher,median_age,pct_white_non_hispanic,pct_black_non_hispanic,pct_native_american_non_hispanic,pct_asian_non_hispanic,pct_pacific_islander_non_hispanic,pct_other_race_non_hispanic,pct_two_or_more_races_non_hispanic,pct_hispanic_or_latino,sex_ratio,pct_male,pct_female
132,ZCTA5 01001,01001,16234.0,75342.0,0.062429,0.357550,46.2,0.833744,0.041764,0.00000,0.030676,0.0,0.003080,0.024640,0.066096,101.564440,0.503881,0.496119
133,ZCTA5 01002,01002,26370.0,70818.0,0.226213,0.753979,27.0,0.689723,0.044672,0.00019,0.098483,0.0,0.001176,0.063671,0.102086,99.214323,0.498028,0.501972
134,ZCTA5 01003,01003,13510.0,NaN,0.211111,0.471698,19.7,0.604219,0.055588,0.00000,0.212435,0.0,0.000000,0.064915,0.062842,84.260775,0.457291,0.542709
135,ZCTA5 01005,01005,5204.0,111148.0,0.087502,0.336558,43.4,0.910069,0.019216,0.00000,0.000000,0.0,0.000000,0.013259,0.057456,95.639098,0.488855,0.511145
136,ZCTA5 01007,01007,15419.0,108125.0,0.085133,0.473048,43.8,0.838187,0.020364,0.00240,0.021467,0.0,0.012712,0.032946,0.071924,89.492442,0.472274,0.527726


In [5]:
from google.colab import files

# 1. Convert the DataFrame to a CSV file
final_df.to_csv('census_data.csv', index=False)

# 2. Download the CSV file to your local machine
files.download('census_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
final_df.head()

,NAME,zip_code,total_population,median_household_income,poverty_rate,pct_bachelors_or_higher,median_age,pct_white_non_hispanic,pct_black_non_hispanic,pct_native_american_non_hispanic,pct_asian_non_hispanic,pct_pacific_islander_non_hispanic,pct_other_race_non_hispanic,pct_two_or_more_races_non_hispanic,pct_hispanic_or_latino,sex_ratio,pct_male,pct_female
132,ZCTA5 01001,01001,16234.0,75342.0,0.062429,0.357550,46.2,0.833744,0.041764,0.00000,0.030676,0.0,0.003080,0.024640,0.066096,101.564440,0.503881,0.496119
133,ZCTA5 01002,01002,26370.0,70818.0,0.226213,0.753979,27.0,0.689723,0.044672,0.00019,0.098483,0.0,0.001176,0.063671,0.102086,99.214323,0.498028,0.501972
134,ZCTA5 01003,01003,13510.0,NaN,0.211111,0.471698,19.7,0.604219,0.055588,0.00000,0.212435,0.0,0.000000,0.064915,0.062842,84.260775,0.457291,0.542709
135,ZCTA5 01005,01005,5204.0,111148.0,0.087502,0.336558,43.4,0.910069,0.019216,0.00000,0.000000,0.0,0.000000,0.013259,0.057456,95.639098,0.488855,0.511145
136,ZCTA5 01007,01007,15419.0,108125.0,0.085133,0.473048,43.8,0.838187,0.020364,0.00240,0.021467,0.0,0.012712,0.032946,0.071924,89.492442,0.472274,0.527726


In [7]:
final_df.isna().sum()

,0
NAME,0
zip_code,0
total_population,0
median_household_income,40
poverty_rate,10
pct_bachelors_or_higher,8
median_age,11
pct_white_non_hispanic,7
pct_black_non_hispanic,7
pct_native_american_non_hispanic,7


In [8]:
duplicates = final_df[final_df.duplicated()]
duplicates

,NAME,zip_code,total_population,median_household_income,poverty_rate,pct_bachelors_or_higher,median_age,pct_white_non_hispanic,pct_black_non_hispanic,pct_native_american_non_hispanic,pct_asian_non_hispanic,pct_pacific_islander_non_hispanic,pct_other_race_non_hispanic,pct_two_or_more_races_non_hispanic,pct_hispanic_or_latino,sex_ratio,pct_male,pct_female


In [10]:
sum = (
    final_df['pct_white_non_hispanic']
    + final_df['pct_asian_non_hispanic']
    + final_df['pct_black_non_hispanic']
    + final_df['pct_hispanic_or_latino']
    + final_df['pct_native_american_non_hispanic']
    + final_df['pct_other_race_non_hispanic']
    + final_df['pct_pacific_islander_non_hispanic']
    + final_df['pct_two_or_more_races_non_hispanic']
)
sum

,0
132,1.0
133,1.0
134,1.0
135,1.0
136,1.0
...,...
747,1.0
748,1.0
749,1.0
750,1.0
